# Photo-z Formalism for Excursion Set Functions

This notebook provides detailed documentation of the photometric redshift (photo-z) extension to the excursion-set theory for void statistics. We present the key equations, derivations, and visualizations of the different components.

## Table of Contents
1. Introduction to Photo-z Effects
2. Tophat Window Functions
3. Angular Damping Factor G(a)
4. Effective Variance with Photo-z
5. Photo-z Barrier Function
6. Multiplicity Functions
7. Void Size Functions

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import pandas as pd

# Import photo-z module
from excursion_set_functions.python import photo_z as pz
from excursion_set_functions.python import integration as int_py

plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 14

---

## 1. Introduction to Photo-z Effects

### Background

Photometric redshifts (photo-z) introduce uncertainties in the line-of-sight (LOS) positions of galaxies. This uncertainty effectively smooths the density field along the LOS, modifying the statistics of density fluctuations and thus the void size function.

The key physical effect is that photo-z errors introduce a damping factor $G(k\sigma_\chi)$ that suppresses power on small scales (large $k$), where $\sigma_\chi$ is the photo-z scatter converted to comoving distance units:

$$\sigma_\chi = \frac{c}{H(z)} \frac{\sigma_{z,\text{photo}}}{1+z}$$

where:
- $c$ is the speed of light
- $H(z)$ is the Hubble parameter at redshift $z$
- $\sigma_{z,\text{photo}}$ is the photometric redshift error

In [ ]:
# Load power spectrum for examples
k_Pk = pd.read_csv('LCDM_matterpower.dat', sep='\\s+', header=0).values
k = k_Pk[:, 0]
Pk = k_Pk[:, 1]

# Define radius array
R = 10 ** np.linspace(-0.5, 2.3, 200)

print(f"Power spectrum: k ∈ [{k.min():.2e}, {k.max():.2e}] h/Mpc")
print(f"Radii: R ∈ [{R.min():.2f}, {R.max():.2f}] Mpc/h")

---

## 2. Tophat Window Functions

### Definition (Equations 85-86)

The tophat window function in Fourier space is:

$$W_T(x) = \frac{3(\sin x - x\cos x)}{x^3} = 3 \frac{j_1(x)}{x}$$

where $j_1(x)$ is the spherical Bessel function of order 1 and $x = kR$.

### First Derivative

$$\frac{dW_T}{dx} = 3\left[\frac{\sin x}{x^2} - \frac{3(\sin x - x\cos x)}{x^4}\right]$$

### Second Derivative

$$\frac{d^2W_T}{dx^2} = 3\left[\frac{\cos x}{x^2} - \frac{5\sin x}{x^3} + \frac{12(\sin x - x\cos x)}{x^5}\right]$$

### Taylor Expansion (for small $x$)

For numerical stability near $x = 0$:

$$W_T(x) \approx 1 - \frac{x^2}{10} + \frac{x^4}{280} - \frac{x^6}{15120} + \mathcal{O}(x^8)$$

In [ ]:
# Plot tophat window function and derivatives
x = np.linspace(0.01, 20, 500)

W = pz.tophat_window(x)
dW = pz.tophat_window_derivative(x)
d2W = pz.tophat_window_second_derivative(x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# W(x)
ax = axes[0]
ax.plot(x, W, 'b-', lw=2)
ax.axhline(0, ls='--', color='gray', lw=0.5)
ax.set_xlabel('$x = kR$')
ax.set_ylabel('$W_T(x)$')
ax.set_title('Tophat Window Function')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 20])

# dW/dx
ax = axes[1]
ax.plot(x, dW, 'g-', lw=2)
ax.axhline(0, ls='--', color='gray', lw=0.5)
ax.set_xlabel('$x = kR$')
ax.set_ylabel("$W'_T(x)$")
ax.set_title('First Derivative')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 20])

# d²W/dx²
ax = axes[2]
ax.plot(x, d2W, 'r-', lw=2)
ax.axhline(0, ls='--', color='gray', lw=0.5)
ax.set_xlabel('$x = kR$')
ax.set_ylabel("$W''_T(x)$")
ax.set_title('Second Derivative')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 20])

plt.tight_layout()
plt.show()

### Window Function Squared

The variance integral involves $W_T^2(kR)$:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = np.linspace(0.01, 30, 500)
W = pz.tophat_window(x)

ax.plot(x, W**2, 'b-', lw=2, label='$W_T^2(x)$')
ax.fill_between(x, 0, W**2, alpha=0.2)
ax.axhline(0, ls='--', color='gray', lw=0.5)
ax.set_xlabel('$x = kR$')
ax.set_ylabel('$W_T^2(x)$')
ax.set_title('Tophat Window Function Squared (enters variance integral)')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 30])
ax.set_ylim([-0.05, 1.05])
ax.legend()

plt.tight_layout()
plt.show()

---

## 3. Angular Damping Factor G(a)

### Definition (Equation 87)

The angular damping factor due to photo-z uncertainty is:

$$G(a) = \frac{\sqrt{\pi}}{2a} \text{erf}(a)$$

where $a = k\sigma_\chi$ and $\text{erf}(x)$ is the error function.

### Properties

- **$G(0) = 1$**: No damping in the spectroscopic limit
- **$G(a \to \infty) \to 0$**: Strong damping at high $k$ (small scales)

### Taylor Expansion (for small $a$)

$$G(a) \approx 1 - \frac{a^2}{3} + \frac{a^4}{10} + \mathcal{O}(a^6)$$

In [ ]:
# Plot G(a) for different arguments
a = np.linspace(0.001, 5, 500)
G = pz.G_photo_z(a)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# G(a) vs a
ax = axes[0]
ax.plot(a, G, 'b-', lw=2)
ax.axhline(1, ls='--', color='gray', lw=1, label='$G(0) = 1$')
ax.set_xlabel('$a = k\\sigma_\\chi$')
ax.set_ylabel('$G(a)$')
ax.set_title('Angular Damping Factor $G(a) = \\frac{\\sqrt{\\pi}}{2a}\\text{erf}(a)$')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xlim([0, 5])
ax.set_ylim([0, 1.1])

# G(k*sigma_chi) for different sigma_chi values
ax = axes[1]
sigma_chi_vals = [0, 10, 30, 50, 100]
colors = ['black', 'blue', 'green', 'orange', 'red']

for sigma_chi, c in zip(sigma_chi_vals, colors):
    if sigma_chi == 0:
        G_k = np.ones_like(k)
    else:
        G_k = pz.G_photo_z(k * sigma_chi)
    ax.plot(k, G_k, color=c, lw=2, label=f'$\\sigma_\\chi$ = {sigma_chi} Mpc/h')

ax.set_xscale('log')
ax.set_xlabel('$k$ [h/Mpc]')
ax.set_ylabel('$G(k\\sigma_\\chi)$')
ax.set_title('Damping Factor vs Wavenumber')
ax.grid(True, alpha=0.3, which='both')
ax.legend()
ax.set_ylim([0, 1.1])

plt.tight_layout()
plt.show()

---

## 4. Effective Variance with Photo-z

### Effective Variance (Equation 88)

The effective variance with photo-z damping is:

$$S_{\text{eff}}(R) = \frac{1}{2\pi^2} \int dk\, k^2 P(k) W_T^2(kR) G(k\sigma_\chi)$$

The damping factor $G$ reduces the contribution from high-$k$ modes.

### First Derivative (Equation 89)

$$\frac{dS_{\text{eff}}}{dR} = \frac{1}{\pi^2} \int dk\, k^3 P(k) W_T(kR) W'_T(kR) G(k\sigma_\chi)$$

### Second Derivative (Equation 90)

$$\frac{d^2S_{\text{eff}}}{dR^2} = \frac{1}{\pi^2} \int dk\, k^2 P(k) \left[(kW'_T)^2 + W_T(k^2W''_T)\right] G(k\sigma_\chi)$$

### Diffusion Coefficient (Equation 91)

$$D_W(R) = \frac{d^2S_{\text{eff}}/dR^2}{(dS_{\text{eff}}/dR)^2}$$

In [ ]:
# Compute Seff for different sigma_chi values
sigma_chi_vals = [0, 10, 30, 50, 100]
colors = ['black', 'blue', 'green', 'orange', 'red']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Seff(R)
ax = axes[0, 0]
for sigma_chi, c in zip(sigma_chi_vals, colors):
    Seff = pz.Seff_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    ax.plot(R, Seff, color=c, lw=2, label=f'$\\sigma_\\chi$ = {sigma_chi} Mpc/h')

ax.set_xscale('log')
ax.set_xlabel('R [Mpc/h]')
ax.set_ylabel('$S_{\\text{eff}}(R)$')
ax.set_title('Effective Variance $S_{\\text{eff}}(R)$')
ax.grid(True, alpha=0.3)
ax.legend()

# dSeff/dR
ax = axes[0, 1]
for sigma_chi, c in zip(sigma_chi_vals, colors):
    dSeff = pz.dSeff_dR_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    ax.plot(R, np.abs(dSeff), color=c, lw=2, label=f'$\\sigma_\\chi$ = {sigma_chi} Mpc/h')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('R [Mpc/h]')
ax.set_ylabel('$|dS_{\\text{eff}}/dR|$')
ax.set_title('First Derivative $|dS_{\\text{eff}}/dR|$')
ax.grid(True, alpha=0.3, which='both')
ax.legend()

# d²Seff/dR²
ax = axes[1, 0]
for sigma_chi, c in zip(sigma_chi_vals, colors):
    d2Seff = pz.d2Seff_dR2_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    ax.plot(R, d2Seff, color=c, lw=2, label=f'$\\sigma_\\chi$ = {sigma_chi} Mpc/h')

ax.set_xscale('log')
ax.set_xlabel('R [Mpc/h]')
ax.set_ylabel('$d^2S_{\\text{eff}}/dR^2$')
ax.set_title('Second Derivative $d^2S_{\\text{eff}}/dR^2$')
ax.grid(True, alpha=0.3)
ax.legend()

# Diffusion coefficient DW
ax = axes[1, 1]
for sigma_chi, c in zip(sigma_chi_vals, colors):
    DW = pz.DW_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    ax.plot(R, DW, color=c, lw=2, label=f'$\\sigma_\\chi$ = {sigma_chi} Mpc/h')

ax.set_xscale('log')
ax.set_xlabel('R [Mpc/h]')
ax.set_ylabel('$D_W(R)$')
ax.set_title('Diffusion Coefficient $D_W = (d^2S/dR^2)/(dS/dR)^2$')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

### Ratio of Effective to Spectroscopic Variance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

S_spectro = pz.Seff_photo_z(Pk, k, R, sigma_chi=0.0)

for sigma_chi, c in zip(sigma_chi_vals[1:], colors[1:]):
    Seff = pz.Seff_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    ratio = Seff / S_spectro
    ax.plot(R, ratio, color=c, lw=2, label=f'$\\sigma_\\chi$ = {sigma_chi} Mpc/h')

ax.axhline(1.0, ls='--', color='black', lw=1)
ax.set_xscale('log')
ax.set_xlabel('R [Mpc/h]')
ax.set_ylabel('$S_{\\text{eff}} / S$')
ax.set_title('Ratio of Effective to Spectroscopic Variance')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim([0, 1.1])

plt.tight_layout()
plt.show()

---

## 5. Photo-z Barrier Function

### Barrier Modification (Equation 93)

The photo-z modified barrier is:

$$B_{\text{ph}}(R) = \alpha_B \sqrt{\frac{S_{\text{eff}}(R)}{S(R)}} B(S(R)) + \beta_B$$

where:
- $B(S)$ is the spectroscopic barrier (e.g., moving barrier for voids)
- $\alpha_B = 1$ and $\beta_B = 0$ for parameter-free theory prediction

### Physical Interpretation

The factor $\sqrt{S_{\text{eff}}/S}$ accounts for the change in the density field statistics due to photo-z smoothing:
- When $\sigma_\chi \to 0$: $S_{\text{eff}} \to S$ and $B_{\text{ph}} \to B$
- When $\sigma_\chi > 0$: $S_{\text{eff}} < S$ and $|B_{\text{ph}}| < |B|$ (reduced barrier height)

In [ ]:
# Define moving barrier parameters
delta_v_lin = -0.8
alpha_barrier = 0.517 * abs(delta_v_lin) - 0.089
beta_barrier = 0.098 * abs(delta_v_lin) + 0.103
gamma_barrier = 0.87

# Spectroscopic quantities
S_spectro = pz.Seff_photo_z(Pk, k, R, sigma_chi=0.0)
B_spectro = alpha_barrier * (1. + (beta_barrier / S_spectro**0.5)**gamma_barrier)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barrier B_ph(R)
ax = axes[0]
for sigma_chi, c in zip(sigma_chi_vals, colors):
    Seff = pz.Seff_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    Bph = pz.B_photo_z(B_spectro, S_spectro, Seff, alpha_B=1.0, beta_B=0.0)
    ax.plot(R, Bph, color=c, lw=2, label=f'$\\sigma_\\chi$ = {sigma_chi} Mpc/h')

ax.set_xscale('log')
ax.set_xlabel('R [Mpc/h]')
ax.set_ylabel('$B_{\\text{ph}}(R)$')
ax.set_title('Photo-z Modified Barrier')
ax.grid(True, alpha=0.3)
ax.legend()

# Barrier as function of S
ax = axes[1]
for sigma_chi, c in zip(sigma_chi_vals, colors):
    Seff = pz.Seff_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    Bph = pz.B_photo_z(B_spectro, S_spectro, Seff, alpha_B=1.0, beta_B=0.0)
    ax.plot(Seff, Bph, color=c, lw=2, label=f'$\\sigma_\\chi$ = {sigma_chi} Mpc/h')

ax.set_xlabel('$S_{\\text{eff}}$')
ax.set_ylabel('$B_{\\text{ph}}$')
ax.set_title('Photo-z Barrier vs Effective Variance')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xlim([0, 2])

plt.tight_layout()
plt.show()

---

## 6. Multiplicity Functions

### Musso-Sheth (MB) Approximation (Equations 95-96)

The photo-z multiplicity using the MB approximation is:

$$f_{\text{ph}}(S_{\text{eff}}) \approx \frac{|T_{\text{ph}}(S_{\text{eff}})|}{\sqrt{2\pi S_{\text{eff}}^3}} \exp\left(-\frac{B_{\text{ph}}^2}{2(1+D_{\text{tot}})S_{\text{eff}}}\right)$$

where the Taylor expansion coefficient:

$$T_{\text{ph}}(S_{\text{eff}}) \approx B_{\text{ph}} - S_{\text{eff}} \frac{dB_{\text{ph}}}{dS_{\text{eff}}}$$

and $D_{\text{tot}} = D_B + D_{\text{ph}}$ is the total diffusion coefficient.

### Upcrossing Formula (Equation 98)

An alternative formulation based on level-crossing statistics:

$$f_{\text{ph}} \approx \frac{e^{-B_{\text{ph}}^2/(2S_{\text{eff}})}}{\sqrt{2\pi S_{\text{eff}}}} \left[\sqrt{\frac{\Gamma_{\text{eff}}}{2\pi S_{\text{eff}}}} e^{-S_{\text{eff}}\beta_*^2/(2\Gamma_{\text{eff}})} + \frac{\beta_*}{2}(\text{erf}(...)+1)\right]$$

where $\beta_* = B_{\text{ph}}/(2S_{\text{eff}}) - dB_{\text{ph}}/dS_{\text{eff}}$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lagrangian multiplicity f(S)
ax = axes[0]
for sigma_chi, c in zip(sigma_chi_vals, colors):
    Seff = pz.Seff_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    dSeff_dR = pz.dSeff_dR_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    
    if sigma_chi == 0:
        dS_dR_spectro = dSeff_dR
    else:
        dS_dR_spectro = pz.dSeff_dR_photo_z(Pk, k, R, sigma_chi=0.0)
    
    B_s = alpha_barrier * (1. + (beta_barrier / Seff**0.5)**gamma_barrier)
    dB_dS = -0.5 * alpha_barrier * beta_barrier**gamma_barrier * gamma_barrier * Seff**(-gamma_barrier/2. - 1)
    
    Bph = pz.B_photo_z(B_s, S_spectro, Seff, alpha_B=1.0, beta_B=0.0)
    dBph_dSeff = pz.dB_photo_z_dSeff(B_s, S_spectro, Seff, dS_dR_spectro, dSeff_dR, dB_dS)
    
    f_Lagr = pz.f_photo_z_MB(Seff, Bph, dBph_dSeff, D_tot=0.0)
    ax.plot(Seff, f_Lagr, color=c, lw=2, label=f'$\\sigma_\\chi$ = {sigma_chi} Mpc/h')

ax.set_xlabel('$S_{\\text{eff}}$')
ax.set_ylabel('$f_{\\text{ph}}(S_{\\text{eff}})$')
ax.set_title('Lagrangian Multiplicity $f(S_{\\text{eff}})$')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xlim([0, 2])

# Lagrangian multiplicity f(R)
ax = axes[1]
for sigma_chi, c in zip(sigma_chi_vals, colors):
    Seff = pz.Seff_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    dSeff_dR = pz.dSeff_dR_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    
    if sigma_chi == 0:
        dS_dR_spectro = dSeff_dR
    else:
        dS_dR_spectro = pz.dSeff_dR_photo_z(Pk, k, R, sigma_chi=0.0)
    
    B_s = alpha_barrier * (1. + (beta_barrier / Seff**0.5)**gamma_barrier)
    dB_dS = -0.5 * alpha_barrier * beta_barrier**gamma_barrier * gamma_barrier * Seff**(-gamma_barrier/2. - 1)
    
    Bph = pz.B_photo_z(B_s, S_spectro, Seff, alpha_B=1.0, beta_B=0.0)
    dBph_dSeff = pz.dB_photo_z_dSeff(B_s, S_spectro, Seff, dS_dR_spectro, dSeff_dR, dB_dS)
    
    f_Lagr = pz.f_photo_z_MB(Seff, Bph, dBph_dSeff, D_tot=0.0)
    f_R = f_Lagr * np.abs(dSeff_dR)
    ax.plot(R, f_R, color=c, lw=2, label=f'$\\sigma_\\chi$ = {sigma_chi} Mpc/h')

ax.set_xscale('log')
ax.set_xlabel('R [Mpc/h]')
ax.set_ylabel('$f(R) = f(S_{\\text{eff}}) |dS_{\\text{eff}}/dR|$')
ax.set_title('Lagrangian Multiplicity $f(R)$')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

---

## 7. Void Size Functions

### Lagrangian VSF

The Lagrangian void size function is simply the multiplicity $f(R)$.

### Eulerian VSF (Equations 97, 99)

The Eulerian (observed) void size function is:

$$\frac{dn_E}{dR_E} = \frac{3}{4\pi R_E^3} f_{\text{ph}}(S_{\text{eff}}) |dS_{\text{eff}}/dR| \frac{dR}{dR_E}$$

where $R_E = q R$ with $q \approx 1.7$ for voids (expansion factor from Lagrangian to Eulerian radius).

In [ ]:
# Compute Eulerian VSF
q = 1.7  # Expansion factor

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Eulerian VSF
ax = axes[0]
for sigma_chi, c in zip(sigma_chi_vals, colors):
    Seff = pz.Seff_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    dSeff_dR = pz.dSeff_dR_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    
    if sigma_chi == 0:
        dS_dR_spectro = dSeff_dR
    else:
        dS_dR_spectro = pz.dSeff_dR_photo_z(Pk, k, R, sigma_chi=0.0)
    
    B_s = alpha_barrier * (1. + (beta_barrier / Seff**0.5)**gamma_barrier)
    dB_dS = -0.5 * alpha_barrier * beta_barrier**gamma_barrier * gamma_barrier * Seff**(-gamma_barrier/2. - 1)
    
    Bph = pz.B_photo_z(B_s, S_spectro, Seff, alpha_B=1.0, beta_B=0.0)
    dBph_dSeff = pz.dB_photo_z_dSeff(B_s, S_spectro, Seff, dS_dR_spectro, dSeff_dR, dB_dS)
    
    f_Lagr = pz.f_photo_z_MB(Seff, Bph, dBph_dSeff, D_tot=0.0)
    
    RE = q * R
    dn_dRE = pz.dnE_dRE_photo_z(f_Lagr, Seff, dSeff_dR, R, RE, 1.0/q)
    
    ax.plot(RE, dn_dRE, color=c, lw=2, label=f'$\\sigma_\\chi$ = {sigma_chi} Mpc/h')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('$R_E$ [Mpc/h]')
ax.set_ylabel('$dn/dR_E$ [(Mpc/h)$^{-4}$]')
ax.set_title('Eulerian Void Size Function')
ax.grid(True, alpha=0.3, which='both')
ax.legend()

# Ratio to spectroscopic
ax = axes[1]
dn_dRE_ref = None
for sigma_chi, c in zip(sigma_chi_vals, colors):
    Seff = pz.Seff_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    dSeff_dR = pz.dSeff_dR_photo_z(Pk, k, R, sigma_chi=sigma_chi)
    
    if sigma_chi == 0:
        dS_dR_spectro = dSeff_dR
    else:
        dS_dR_spectro = pz.dSeff_dR_photo_z(Pk, k, R, sigma_chi=0.0)
    
    B_s = alpha_barrier * (1. + (beta_barrier / Seff**0.5)**gamma_barrier)
    dB_dS = -0.5 * alpha_barrier * beta_barrier**gamma_barrier * gamma_barrier * Seff**(-gamma_barrier/2. - 1)
    
    Bph = pz.B_photo_z(B_s, S_spectro, Seff, alpha_B=1.0, beta_B=0.0)
    dBph_dSeff = pz.dB_photo_z_dSeff(B_s, S_spectro, Seff, dS_dR_spectro, dSeff_dR, dB_dS)
    
    f_Lagr = pz.f_photo_z_MB(Seff, Bph, dBph_dSeff, D_tot=0.0)
    
    RE = q * R
    dn_dRE = pz.dnE_dRE_photo_z(f_Lagr, Seff, dSeff_dR, R, RE, 1.0/q)
    
    if sigma_chi == 0:
        dn_dRE_ref = dn_dRE
    else:
        ratio = dn_dRE / dn_dRE_ref
        ax.plot(RE, ratio, color=c, lw=2, label=f'$\\sigma_\\chi$ = {sigma_chi} Mpc/h')

ax.axhline(1.0, ls='--', color='black', lw=1, label='Spectroscopic')
ax.set_xscale('log')
ax.set_xlabel('$R_E$ [Mpc/h]')
ax.set_ylabel('Ratio to spectroscopic')
ax.set_title('Relative Effect of Photo-z on VSF')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_ylim([0, 1.5])

plt.tight_layout()
plt.show()

---

## Summary

### Key Equations Reference

| Quantity | Equation |
|----------|----------|
| Tophat window | $W_T(x) = 3(\sin x - x\cos x)/x^3$ |
| Angular damping | $G(a) = \sqrt{\pi}/(2a) \cdot \text{erf}(a)$ |
| Effective variance | $S_{\text{eff}} = \int dk\, k^2 P(k) W_T^2(kR) G(k\sigma_\chi) / (2\pi^2)$ |
| Photo-z barrier | $B_{\text{ph}} = \sqrt{S_{\text{eff}}/S} \cdot B(S)$ |
| Multiplicity (MB) | $f_{\text{ph}} \propto |T_{\text{ph}}|/\sqrt{S_{\text{eff}}^3} \exp(-B_{\text{ph}}^2/(2S_{\text{eff}}))$ |
| Eulerian VSF | $dn/dR_E = (3/4\pi R_E^3) f_{\text{ph}} |dS_{\text{eff}}/dR| (dR/dR_E)$ |

### Physical Effects of Photo-z

1. **Damping**: Photo-z uncertainties damp small-scale power via $G(k\sigma_\chi)$
2. **Reduced Variance**: $S_{\text{eff}} < S$ when $\sigma_\chi > 0$
3. **Modified Barrier**: Barrier height reduced by $\sqrt{S_{\text{eff}}/S}$
4. **Scale Dependence**: Small voids more affected than large voids

### Module Structure

The `photo_z` submodule is organized as:
- `window.py`: Tophat window functions
- `angular.py`: Angular damping factor
- `variance.py`: Effective variance calculations
- `barrier.py`: Barrier function modifications
- `multiplicity.py`: Multiplicity functions
- `vsf.py`: Complete VSF pipeline